In [ ]:
import os
os.system("git clone https://github.com/comfyanonymous/ComfyUI.git && cd ComfyUI && git checkout 5b80adda")

In [ ]:
import os
import subprocess

os.chdir("ComfyUI")

# Now safely install remaining dependencies
subprocess.run("pip install -r requirements.txt", shell=True)

In [ ]:
import subprocess

print("Downloading models...")
result = subprocess.run(
    "wget -O models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors && "
    "wget -O models/vae/wan_2.1_vae.safetensors https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors && "
    "wget -O models/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors && "
    "wget -O models/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors && "
    "wget -O models/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors && "
    "wget -O models/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors && "
    "wget -O models/diffusion_models/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors && "
    "wget -O models/diffusion_models/wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors && "
    "wget -O models/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_low_noise.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_low_noise.safetensors && "
    "wget -O models/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_high_noise.safetensors https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_high_noise.safetensors",
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Models downloaded")
else:
    print(f"✗ Models error: {result.stderr}")

print("Downloading workflows...")
result = subprocess.run(
    "cd ComfyUI && mkdir -p user/default/workflows && "
    "wget -O user/default/workflows/wan22_t2v_14B.json https://raw.githubusercontent.com/ziyaad30/wan_notebooks/refs/heads/main/wan22_t2v_14B.json && "
    "wget -O user/default/workflows/wan22_i2v_14B.json https://raw.githubusercontent.com/ziyaad30/wan_notebooks/refs/heads/main/wan22_i2v_14B.json",
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print("✓ Workflows downloaded")
else:
    print(f"✗ Workflows error: {result.stderr}")

In [ ]:
import subprocess, os
os.chdir("/workspace/ComfyUI")
subprocess.Popen(["python3", "main.py", "--listen", "0.0.0.0", "--port", "8000"], 
                 start_new_session=True, stdout=open("comfyui.log", "w"), stderr=subprocess.STDOUT)

In [ ]:
import subprocess, time, os, re

time.sleep(5)

# Start Pinggy tunnel - the +force option will kill any existing connection
subprocess.Popen(
    ["ssh", "-p", "443",
     "-R0:localhost:8000",
     "-o", "StrictHostKeyChecking=no",
     "-o", "ServerAliveInterval=30",
     "UYekzW20QmT+force@a.pinggy.io"],  # <-- The key fix: "+force"
    start_new_session=True,
    stdout=open("/workspace/tunnel.log", "w"),
    stderr=subprocess.STDOUT
)

# Extract the URL from the log
for i in range(20):
    time.sleep(1)
    if os.path.exists("/workspace/tunnel.log"):
        with open("/workspace/tunnel.log") as f:
            log = f.read()
            match = re.search(r'(https://[^\s]+\.pinggy-free\.link)', log)
            if match:
                url = match.group(1)
                print(f"URL: {url}")
                with open("/workspace/ngrok_url.txt", "w") as uf:
                    uf.write(url)
                break